Khushi Khatri BEB222 EXPERIMENT 4      
Aim:Perform morphological analysis and word generation for any given text

Theory:
Morphological processing forms the computational backbone of natural language processing by managing how words are structured, parsed, and synthesized. Morphological analysis is the deductive process of breaking a complex surface word down into its smallest meaningful constituent parts, known as morphemes. When a system analyzes a text, it first tokenizes the sentence into individual words and then strips away superficial variations to uncover the core dictionary form, known as the lemma. Along with finding this base lemma, the analyzer maps the word's grammatical features—such as part of speech, tense, number, gender, and case—creating an explicit linguistic profile for every token.

Word generation, or morphological synthesis, acts as the exact reverse process by transforming a base lemma back into a grammatically correct surface word. The generation engine takes two primary inputs: the root dictionary word and a targeted set of grammatical requirements, such as forcing a verb into the progressive past tense. To output a real word, the system must navigate morphotactic constraints, which dictate the legal structural ordering of prefixes, roots, and suffixes in a specific language. Finally, the system applies orthographic and phonological adjustment rules to handle spelling boundary mutations, ensuring that a combination like "study" plus the past tense features successfully resolves to "studied" rather than an ungrammatical spelling.

In [2]:
import re
import pandas as pd
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()
print('Setup complete.')

Setup complete.


[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/computer/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/computer/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


In [4]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def morphological_analysis(text):
    tokens = [tok for tok in word_tokenize(text) if tok.isalpha()]
    pos_tags = nltk.pos_tag(tokens)
    results = []
    for word, tag in pos_tags:
        lemma = lemmatizer.lemmatize(word.lower(), get_wordnet_pos(tag))
        results.append({'word': word, 'lemma': lemma, 'pos_tag': tag})
    return pd.DataFrame(results)

sample_text = df['review_text'].iloc[0]
print('Original Text:\n', sample_text, '\n')

analysis_df = morphological_analysis(sample_text)
analysis_df

Original Text:
 Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed. 



,word,lemma,pos_tag
0,Amazing,amaze,VBG
1,stay,stay,VB
2,The,the,DT
3,place,place,NN
4,felt,felt,VBD
5,very,very,RB
6,cozy,cozy,JJ
7,for,for,IN
8,guests,guest,NNS
9,was,be,VBD


In [5]:
COMMON_PREFIXES = ['un', 're', 'dis', 'mis', 'pre', 'non', 'in', 'im', 'over', 'under']
COMMON_SUFFIXES = ['ing', 'ed', 'ly', 'ness', 'tion', 'sion', 'ment', 'ful', 'less', 'able', 'ible', 's', 'es']

def split_morphemes(word):
    original = word
    prefix, suffix = '', ''
    for p in sorted(COMMON_PREFIXES, key=len, reverse=True):
        if word.startswith(p) and len(word) - len(p) > 2:
            prefix = p
            word = word[len(p):]
            break
    for s in sorted(COMMON_SUFFIXES, key=len, reverse=True):
        if word.endswith(s) and len(word) - len(s) > 2:
            suffix = s
            word = word[:-len(s)]
            break
    return {'word': original, 'prefix': prefix, 'root': word, 'suffix': suffix}

test_words = ['unhappiness', 'replayed', 'disconnecting', 'careless', 'national', 'checking']
morpheme_df = pd.DataFrame([split_morphemes(w) for w in test_words])
morpheme_df

,word,prefix,root,suffix
0,unhappiness,un,happi,ness
1,replayed,re,play,ed
2,disconnecting,dis,connect,ing
3,careless,,care,less
4,national,,national,
5,checking,,check,ing


In [6]:
def generate_inflection(root, feature):
    irregular_plurals = {'child': 'children', 'man': 'men', 'woman': 'women', 'mouse': 'mice'}
    irregular_past = {'go': 'went', 'run': 'ran', 'eat': 'ate', 'see': 'saw', 'buy': 'bought'}

    if feature == 'plural':
        if root in irregular_plurals:
            return irregular_plurals[root]
        elif root.endswith(('s', 'x', 'z', 'ch', 'sh')):
            return root + 'es'
        elif root.endswith('y') and root[-2] not in 'aeiou':
            return root[:-1] + 'ies'
        else:
            return root + 's'
    elif feature == 'past':
        if root in irregular_past:
            return irregular_past[root]
        elif root.endswith('e'):
            return root + 'd'
        elif root.endswith('y') and root[-2] not in 'aeiou':
            return root[:-1] + 'ied'
        else:
            return root + 'ed'
    elif feature == 'present_participle':
        if root.endswith('e') and root != 'be':
            return root[:-1] + 'ing'
        else:
            return root + 'ing'
    elif feature == 'third_person_singular':
        if root.endswith(('s', 'x', 'z', 'ch', 'sh')):
            return root + 'es'
        elif root.endswith('y') and root[-2] not in 'aeiou':
            return root[:-1] + 'ies'
        else:
            return root + 's'
    else:
        return root

examples = [
    ('play', 'past'), ('play', 'present_participle'), ('play', 'third_person_singular'),
    ('child', 'plural'), ('city', 'plural'), ('go', 'past'), ('box', 'plural')
]
gen_df = pd.DataFrame([{'root': r, 'feature': f, 'generated_word': generate_inflection(r, f)} for r, f in examples])
gen_df

,root,feature,generated_word
0,play,past,played
1,play,present_participle,playing
2,play,third_person_singular,plays
3,child,plural,children
4,city,plural,cities
5,go,past,went
6,box,plural,boxes


In [7]:
DERIVATIONAL_RULES = {
    'ness': lambda root: root + 'ness',
    'ment': lambda root: root + 'ment',
    'tion': lambda root: root[:-1] + 'tion' if root.endswith('e') else root + 'tion',
    'ful':  lambda root: root + 'ful',
    'less': lambda root: root + 'less',
    'able': lambda root: root[:-1] + 'able' if root.endswith('e') else root + 'able',
    'al':   lambda root: root + 'al',
}

def generate_derivation(root, suffix):
    if suffix == 'ness' and root.endswith('y'):
        return root[:-1] + 'iness'
    if suffix in DERIVATIONAL_RULES:
        return DERIVATIONAL_RULES[suffix](root)
    return root + suffix

derivation_examples = [
    ('happy', 'ness'), ('develop', 'ment'), ('educate', 'tion'),
    ('care', 'ful'), ('care', 'less'), ('nation', 'al')
]
deriv_df = pd.DataFrame([{'root': r, 'suffix': s, 'derived_word': generate_derivation(r, s)} for r, s in derivation_examples])
deriv_df

,root,suffix,derived_word
0,happy,ness,happiness
1,develop,ment,development
2,educate,tion,educattion
3,care,ful,careful
4,care,less,careless
5,nation,al,national


In [8]:
def analyze_review_morphology(text):
    if not isinstance(text, str):
        return pd.Series({'lemmas': [], 'pos_tags': []})
    tokens = [tok for tok in word_tokenize(text) if tok.isalpha()]
    pos_tags = nltk.pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(w.lower(), get_wordnet_pos(t)) for w, t in pos_tags]
    tags = [t for _, t in pos_tags]
    return pd.Series({'lemmas': lemmas, 'pos_tags': tags})

sample_df = df.head(200).copy()
morph_results = sample_df['review_text'].apply(analyze_review_morphology)
df_morph = pd.concat([sample_df[['review_id', 'review_text']], morph_results], axis=1)
df_morph.head(10)

,review_id,review_text,lemmas,pos_tags
0,369314882,Amazing stay! The place felt very cozy for 4 g...,"[amaze, stay, the, place, felt, very, cozy, fo...","[VBG, VB, DT, NN, VBD, RB, JJ, IN, NNS, VBD, J..."
1,490116563,It was okay for the price. Location in XIII Au...,"[it, be, okay, for, the, price, location, in, ...","[PRP, VBD, VBN, IN, DT, NN, NNP, IN, NNP, NNP,..."
2,582235668,Loved every minute of it. Our superhost was su...,"[love, every, minute, of, it, our, superhost, ...","[VBN, DT, NN, IN, PRP, PRP$, NN, VBD, JJ, NN, ..."
3,68054683,Decent stay overall. Some things could be impr...,"[decent, stay, overall, some, thing, could, be...","[NNP, NN, RB, DT, NNS, MD, VB, VBN, IN, NN, IN..."
4,248483824,Reasonable for a short trip. Location in Long ...,"[reasonable, for, a, short, trip, location, in...","[JJ, IN, DT, JJ, NN, NNP, IN, NNP, NNP, NNP, V..."
5,155617131,Decent stay overall. It served its purpose for...,"[decent, stay, overall, it, serve, it, purpose...","[NNP, NN, IN, PRP, VBD, PRP$, NN, IN, PRP$, NN..."
6,710244614,"Nothing special, but fine. Our superhost was p...","[nothing, special, but, fine, our, superhost, ...","[VBG, JJ, CC, JJ, PRP$, NN, VBD, JJ, CC, JJ, T..."
7,299174484,We had a rough experience. The location in Enc...,"[we, have, a, rough, experience, the, location...","[PRP, VBD, DT, JJ, NN, DT, NN, IN, VBD, JJR, I..."
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[perfect, for, our, trip, be, smooth, and, the...","[NN, IN, PRP$, NN, VBD, JJ, CC, DT, NNS, VBD, ..."
9,469473761,Would not recommend. The private room in house...,"[would, not, recommend, the, private, room, in...","[MD, RB, VB, DT, JJ, NN, IN, NN, VBD, RB, RB, ..."


In [9]:
df_morph.to_csv('Morphological_Analysis_Airbnb_Reviews.csv', index=False)
print('Saved to Morphological_Analysis_Airbnb_Reviews.csv')

Saved to Morphological_Analysis_Airbnb_Reviews.csv
